In [ ]:
#Question 1: Install Spark and PySpark

In [1]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [5]:
print(spark.version)

3.5.4


In [6]:
#Question 2: Yellow October 2024

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

input_path = 'C:/Tools/tmp/data/raw/yellow/2024/10/yellow_tripdata_2024-10.parquet'

df = spark.read.parquet(input_path)
df = df.repartition(4)

output_path = 'C:/Tools/tmp/homework/2'

df.write.mode("overwrite").parquet(output_path)


In [6]:

df.registerTempTable('trips_data')  

spark.sql("""

SELECT count(1) FROM trips_data 
WHERE DATE(tpep_pickup_datetime) = '2024-10-15';
""").show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



In [4]:

spark.sql("""
SELECT 
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS trip_duration
FROM trips_data
ORDER BY trip_duration DESC
LIMIT 10;
""").show()                  


+------------------+
|     trip_duration|
+------------------+
|162.61777777777777|
|           143.325|
|137.76055555555556|
|114.83472222222223|
| 89.89833333333333|
| 89.44611111111111|
| 70.29916666666666|
| 67.57333333333334|
| 66.06666666666666|
|           46.4225|
+------------------+



In [4]:

lookup_df = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')

lookup_df.registerTempTable('lookup')  

spark.sql("""

SELECT lookup.Zone , count(1) as num_trips FROM trips_data 
INNER JOIN lookup ON lookup.LocationID = trips_data.PULocationID
GROUP BY lookup.Zone
ORDER BY num_trips ASC;
""").show()  


+--------------------+---------+
|                Zone|num_trips|
+--------------------+---------+
|Governor's Island...|        1|
|       Rikers Island|        2|
|       Arden Heights|        2|
|         Jamaica Bay|        3|
| Green-Wood Cemetery|        3|
|Charleston/Totten...|        4|
|   Rossville/Woodrow|        4|
|       Port Richmond|        4|
|Eltingville/Annad...|        4|
|       West Brighton|        4|
|         Great Kills|        6|
|        Crotona Park|        6|
|Heartland Village...|        7|
|     Mariners Harbor|        7|
|Saint George/New ...|        9|
|             Oakwood|        9|
|       Broad Channel|       10|
|New Dorp/Midland ...|       10|
|         Westerleigh|       12|
|     Pelham Bay Park|       12|
+--------------------+---------+
only showing top 20 rows

